# Experiment C1 — CartPole: Teach AI to Balance a Pole 🛒
**AI Builders Lab | Google Colab | Reinforcement Learning**

A cart moves left or right. Can an AI learn to keep a pole upright?

**Run:** Runtime → Run all. **CPU is sufficient.** Training usually takes minutes, but actual time varies.

We use **Proximal Policy Optimization (PPO)**, a reinforcement learning algorithm. This notebook is an **action policy**, not yet a learned world model. Compare it with the Mini-World Model notebook.

## 1. Physics intuition
- **Observation (4 numbers):** cart position, cart velocity, pole angle, pole angular velocity.
- **Action:** 0 = push left, 1 = push right.
- **Reward:** +1 for each time step the pole remains balanced.
- **Episode ends:** if the pole tips too far, cart travels too far, or time limit is reached.

A neural network learns a **policy** $\pi_\theta(a_t\mid s_t)$: what action should be taken given the current state?

Unlike supervised learning, there are **no correct-action labels** in the training set.

In [ ]:
%pip -q install "stable-baselines3>=2.3,<3" "gymnasium>=1.0,<2" "imageio[ffmpeg]"

In [ ]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.utils import set_random_seed

SEED = 42
ENV_ID = "CartPole-v1"
TRAIN_STEPS = 40_000   # Try 10_000, 40_000, or 80_000
VIDEO_FILE = "cartpole_demo.mp4"
VIDEO_MAX_STEPS = 500
set_random_seed(SEED)

env = gym.make(ENV_ID)
observation, info = env.reset(seed=SEED)
print("Observation:", observation)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
env.close()

## 2. Try a random controller
A random policy almost always fails quickly. This gives us a baseline to compare against the trained AI.

In [ ]:
random_env = gym.make(ENV_ID)
scores = []
for episode in range(10):
    obs, _ = random_env.reset(seed=SEED+episode)
    total = 0
    while True:
        obs, reward, terminated, truncated, _ = random_env.step(random_env.action_space.sample())
        total += reward
        if terminated or truncated:
            break
    scores.append(total)
random_env.close()
print("Random policy: mean reward over 10 episodes =", np.mean(scores))

## 3. Train a PPO agent
PPO learns from **trial and error**: observe → act → receive reward → update the policy.

We use a small two-layer MLP. The environment is supplied by Gymnasium; PPO is supplied by Stable-Baselines3.

**Important:** PPO uses PyTorch internally, not TensorFlow. It is chosen here for reliability and readable Colab code.

In [ ]:
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback
import matplotlib.pyplot as plt
import numpy as np

class EpisodeLogger(BaseCallback):
    """Collect completed episode rewards and lengths from Monitor."""
    def __init__(self):
        super().__init__()
        self.rewards = []
        self.lengths = []
    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.rewards.append(float(info["episode"]["r"]))
                self.lengths.append(int(info["episode"]["l"]))
        return True

def plot_learning(log):
    if not log.rewards:
        print("No completed episodes recorded yet.")
        return
    window = min(25, len(log.rewards))
    smoothed = np.convolve(log.rewards, np.ones(window)/window, mode="valid")
    plt.figure(figsize=(9, 4))
    plt.plot(log.rewards, alpha=.25, label="Episode reward")
    plt.plot(np.arange(window-1,len(log.rewards)),smoothed,label=f"{window}-episode average")
    plt.xlabel("Completed training episodes")
    plt.ylabel("Total episode reward")
    plt.title("Reinforcement learning progress")
    plt.legend()
    plt.grid(alpha=.25)
    plt.show()

In [ ]:
train_env = Monitor(gym.make(ENV_ID))
train_env.reset(seed=SEED)
log = EpisodeLogger()
agent = PPO(
    "MlpPolicy", train_env,
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    seed=SEED,
    verbose=0,
    device="cpu",
)
agent.learn(total_timesteps=TRAIN_STEPS, callback=log)
train_env.close()
plot_learning(log)

## 4. Test the learned controller
CartPole-v1 has a 500-step time limit. High average reward means the controller keeps the pole balanced longer; training is stochastic, so results vary.

In [ ]:
from stable_baselines3.common.evaluation import evaluate_policy
eval_env = Monitor(gym.make(ENV_ID))
mean_reward, std_reward = evaluate_policy(agent, eval_env, n_eval_episodes=10, deterministic=True)
print(f"Mean evaluation reward: {mean_reward:.1f} ± {std_reward:.1f}")
eval_env.close()

## 5. Watch your AI balance the pole
Record a short episode and display it directly in Colab. The notebook writes an MP4 in the current Colab runtime.

In [ ]:
from IPython.display import display, HTML
from base64 import b64encode
from pathlib import Path
import imageio.v2 as imageio

def record_episode(agent, env_id, filename, max_steps=1000):
    video_env = gym.make(env_id, render_mode="rgb_array")
    obs, _ = video_env.reset(seed=SEED+1)
    frames = [video_env.render()]
    for _ in range(max_steps):
        action, _ = agent.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = video_env.step(action)
        frames.append(video_env.render())
        if terminated or truncated:
            break
    video_env.close()
    imageio.mimsave(filename, frames, fps=30, macro_block_size=1)
    print("Saved", filename, "frames:", len(frames))
    encoded = b64encode(Path(filename).read_bytes()).decode("ascii")
    display(HTML(f'<video width="640" controls><source src="data:video/mp4;base64,{encoded}" type="video/mp4"></video>'))

record_episode(agent, ENV_ID, VIDEO_FILE, max_steps=VIDEO_MAX_STEPS)

## 6. Save the trained policy
You can download this file from Colab's file sidebar, or train again by rerunning the notebook.

In [ ]:
agent.save("cartpole_ppo_policy")
print("Saved cartpole_ppo_policy.zip")

## 7. Student challenges
1. Set `TRAIN_STEPS = 10_000`. How does the evaluation score change?
2. Try 80,000 steps. Does the reward reliably improve?
3. What is the difference between the **4-dimensional observation** and the **2 possible actions**?
4. Explain why the random policy performs worse.
5. Why is this **reinforcement learning** instead of supervised learning?
6. **World Model connection:** A policy predicts an action; a world model predicts the next state. How could we use a learned world model to choose the best push?

**Key idea:** *Policy = what to do. World model = what happens if I do it.*